In [6]:
using Pkg
Pkg.activate("..")
Pkg.instantiate()

  Activating project at `d:\TFM code`


In [19]:
using Test
include("module/mixed_pure.jl")
nothing

In [4]:
L = 8
μ = 0.6
t = 0.4
r = 20
ψ0, sP, _, _ = purified_domainwall(L, μ)
ts = collect(range(0.0, t; length = r + 1))
nothing

In [20]:
# Main path (local interleaved gate engine expected for per=false)
ψv, Sz, S, steps = MPS_evo(
    sP, ψ0, ts, r;
    cutoff = 1e-10,
    showprogress = true,
    store_states = true,
    snapshot_every = 5,
    sz_every = 2,
    entropy_every = 3
)
nothing

MPS evo (r=500) 100%|████████████████████████████████████| Time: 0:05:59


In [9]:
@test size(Sz) == (r + 1, L)
@test length(S) == r + 1
@test issorted(steps)
@test first(steps) == 1
@test last(steps) == r + 1
@test length(ψv) == length(steps)

for i in 1:(r + 1)
    expect_sz = (i == 1) || (i == r + 1) || (i % 2 == 0)
    expect_s  = (i == 1) || (i == r + 1) || (i % 3 == 0)
    @test expect_sz ? all(isfinite, Sz[i, :]) : all(isnan, Sz[i, :])
    @test expect_s  ? isfinite(S[i]) : isnan(S[i])
end

# Fallback path (per=true)
ev_per = MPS_evo(sP, ψ0, ts, r; per = true, cutoff = 1e-10, showprogress = false, store_states = false)
@test isempty(ev_per[1])
@test isempty(ev_per[4])

# Multi-r wrapper contract
all_evos = Diff_trotter_r(
    sP, ψ0, t, [10, 20];
    cutoff = 1e-10,
    doprint = false,
    showprogress = false,
    store_states = true,
    snapshot_every = 5,
    sz_every = 2,
    entropy_every = 3
)
@test length(all_evos) == 2
@test all(length(ev) == 4 for ev in all_evos)

println("All mixed_pure checks passed.")


All mixed_pure checks passed.


In [12]:
include("module/mixed_pure.jl")

L = 20
μ = 0.6
t = 5.0
r = 500

ψ0, sP, _, _ = purified_domainwall(L, μ)
ts = collect(range(0.0, t; length = r + 1))

ev = MPS_evo(
    sP, ψ0, ts, r;
    cutoff=1e-10,
    showprogress=true,
    store_states=true,   # only needed if you want state-distance plots later
    snapshot_every=1,
    sz_every=1,
    entropy_every=1
)

c = div(L, 2)
tau = t / r

combined, anim = plot_spin_dynamics(ts, ev[2], ev[3], c, L, tau; gifname="Plots/spins.gif", fps=30)
savefig(combined, "Plots/combined.png")

MPS evo (r=500) 100%|████████████████████████████████████| Time: 0:06:26
[ Info: Saved animation to d:\TFM code\Tensor Networks Code\Plots\spins.gif


"d:\\TFM code\\Tensor Networks Code\\Plots\\combined.png"